In [ ]:
import sys
from os.path import join
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
mm = 1.0/2.54/10

repo_root_path = join('..','..')

python_modules_paths = (
    join(repo_root_path, 'src', 'python'),
    'python'
)
for python_modules_path in python_modules_paths:
    if python_modules_path not in sys.path:
        sys.path.append(python_modules_path)

from foam.common import get_wpd_columns
from scm.scm import ilmenite, ilmenite_orig, Abad_SCR, Abad_SCRd, IlmenitePhase, parseReaction, concentration_from_molar_fraction
from labreactor.labreactor import LabReactor
from labreactor.labreactor_analytical import gas_yield_Leion
from labreactor.plotting import plot_labReactor_eff, plot_labReactors_eff, plot_labReactors_exp_eff

plt.style.use(os.path.join(repo_root_path, 'src', 'python', 'clc.mplstyle'))
# TODO: green color (third) is too close to blue (first)
# plt.style.use('tableau-colorblind10')
# ilmenite['act']['H2'].k(T=950+273)
from specie_properties.common import AW, MW

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
def fit_analytical_gamma_omega_T(fuel, x, y, Ro, rhoox, nVolRate_mLn_min, mass, p=101325):
    from scipy.optimize import curve_fit
    
    x_orig = x; y_orig = y
    x = np.linspace(x_orig[0], x_orig[-1], 101)
    y = np.interp(x, x_orig, y_orig)

    vgas_yield_Leion = np.vectorize(gas_yield_Leion)
    f = lambda omega, T: vgas_yield_Leion(
        phase = IlmenitePhase(rhoox=rhoox, Ro=Ro), 
        reaction = ilmenite['act'][fuel], 
        ms=mass, 
        oxidation=1 - (1-omega)/Ro, 
        T=T, 
        p=101325, 
        xA0=1.0, 
        Vdot0=nVolRate_mLn_min * 1e-6 / 60 * T / ( 25 + 273.15 )
    )
    popt, pcov = curve_fit(f, x, y, p0=1000)
    return popt[0]

def fit_analytical_gamma_omega_T_Berguerand(fuel, x, y, mass):
    return fit_analytical_gamma_omega_T(fuel, x, y, Ro=0.033, rhoox=3000, nVolRate_mLn_min=900, mass=mass, p=101325)

def remove_initial_from_validation_data(x, y, limit_fraction=0.8):
    key = 800
    x_new = np.flip(x)
    y_new = np.flip(y)
    inds_high = np.where(y_new > limit_fraction*np.max(y_new))[0]
    if len(inds_high) > 0:
        cutoff_ind = inds_high[-1]
        x_new = x_new[:cutoff_ind]
        y_new = y_new[:cutoff_ind]
    return x_new, y_new

dfs_validation = {
    'H2': pd.read_csv(join('validation',f'Berguerand2011_fig5.csv'), skiprows=2, names=get_wpd_columns(join('validation',f'Berguerand2011_fig5.csv'))),
    'CO': pd.read_csv(join('validation',f'Berguerand2011_fig6.csv'), skiprows=2, names=get_wpd_columns(join('validation',f'Berguerand2011_fig6.csv'))),
}

# TfitdegC = fit_analytical_gamma_omega_T('CO', x_orig, y_orig, 0.033, 3000, 900, 0.003) - 273.15
# print(TfitdegC)
# plt.plot(x_orig, y_orig)


In [ ]:
# TinsdegC = {
#     'act_CO_3g': [770,780,800,820,840,865,870,900,905,910,950,960,1000,1005,1050],
#     'act_CO_6g': [910, 950],  # 915 could not finish, crashed
#     'act_CO': [925, 928, 950],
#     'act_H2_3g': [790,800,830,860,885,888,900,935,950,982,1000,1028,1030,1050],
#     'act_H2_6g': [930,936,950],
#     'act_H2': [935,943,950],
#     'act_CH4': [950, 958, 960],
#     'act_syngas': [930,935,950],
#     'act_O2': [935,940,950],
# }

TinsdegC = {
    'act_CO_3g': [800, 850, 900, 950, 1000, 1050],
    'act_CO_6g': [900, 950, 1000],
    # 'act_CO': [925, 928, 950],
    'act_H2_3g': [800, 850, 900, 950, 1000, 1050],
    'act_H2_6g': [900, 950, 1000],
    # 'act_H2': [935,943,950],
    'act_CH4': [950, 983],
    'act_syngas': [950, 967],
    # 'act_O2': [950, 957],
}

Tincrs = dict()
tends = dict()
for code in TinsdegC.keys():
    if len(code.split('_')) > 2:
        chem = code.split('_')[0]
        fuel = code.split('_')[1]
        mass = code.split('_')[2][:-1]
    else:
        chem = code.split('_')[0]
        fuel = code.split('_')[1]
    print(fuel)
    Tincrs[code] = list(); tends[code] = list()
    for TindegC in TinsdegC[code]:
        lr = LabReactor(f'labReactor_{code}_{TindegC}C')
        t_end = lr.done_time()
        Tin = lr.T
        Tmean = lr.temperature().loc[:t_end].mean()
        Tincr = Tmean - Tin
        Tincrs[code].append(Tincr)
        tends[code].append(t_end)

In [ ]:
Tincrs

In [ ]:
fig, ax = plt.subplots(figsize=[90*mm, 85*mm])

ax.plot(TinsdegC['act_CO_3g'],  Tincrs['act_CO_3g'],    '-s', color=colors[0], label=r'$\mathrm{CO}$, 3 g')
ax.plot(TinsdegC['act_CO_6g'],  Tincrs['act_CO_6g'],    '--x', color=colors[0], label=r'$\mathrm{CO}$, 6 g', markersize=10)
# # ax.plot(TinsdegC['act_CO'],     Tincrs['act_CO'],       '-o', color=colors[0], label=r'$\mathrm{CO}$ 15 g')
ax.plot(TinsdegC['act_H2_3g'],  Tincrs['act_H2_3g'],    '-s', color=colors[1], label=r'$\mathrm{H}_2$, 3 g')
ax.plot(TinsdegC['act_H2_6g'],  Tincrs['act_H2_6g'],    '--x', color=colors[1], label=r'$\mathrm{H}_2$, 6 g', markersize=10)
# # ax.plot(TinsdegC['act_H2'],     Tincrs['act_H2'],       '-o', color=colors[1], label=r'$\mathrm{H}_2$ 15 g')
ax.plot(TinsdegC['act_CH4'],    Tincrs['act_CH4'],      '-o', color=colors[2], label=r'$\mathrm{CH}_4$, 15 g')
ax.plot(TinsdegC['act_syngas'], Tincrs['act_syngas'],   '-o', color=colors[3], label=r'syngas, 15 g')
# # ax.plot(TinsdegC['act_O2'],     Tincrs['act_O2'],       '-o', color=colors[4], label=r'O2 15 g')

# listTexpdegC = [800,900,950,1000,1050]
# Tsexpfit = list()
# for TexpdegC in listTexpdegC:
#     x, y = remove_initial_from_validation_data(
#         dfs_validation['CO'][f'{TexpdegC}C_X'].dropna().to_numpy(), 
#         dfs_validation['CO'][f'{TexpdegC}C_Y'].dropna().to_numpy(), 
#         limit_fraction=0.8)
#     T = fit_analytical_gamma_omega_T_Berguerand('CO', x, y, mass=0.003)
#     Tsexpfit.append(T)
#     print(T)
# ax.plot(listTexpdegC, np.array(Tsexpfit)-273.15-np.array(listTexpdegC))
# print(np.array(Tsexpfit)-273.15-np.array(listTexpdegC))

# ax.legend(loc='upper left', ncols=2, fontsize=7)
ax.legend(loc='upper left', ncols=2)
ax.set(
    xlabel = r'$T_\mathrm{inlet}, ^\circ \mathrm{C}$',
    ylabel = r'$\overline{T}_\mathrm{solids} \minus T_\mathrm{inlet} $, K',
    ylim = [-20,56]
)
fig.tight_layout()
fig.savefig('plots/labReactor_results_dT_vs_T.pdf')

In [ ]:
fig, ax = plt.subplots(figsize=[90*mm, 85*mm])

ax.plot(TinsdegC['act_CO_3g'], tends['act_CO_3g'], '-s', color=colors[0], label=r'$\mathrm{CO}$, 3 g')
ax.plot(TinsdegC['act_CO_6g'], tends['act_CO_6g'], '-x', color=colors[0], label=r'$\mathrm{CO}$, 6 g', markersize=10)
ax.plot(TinsdegC['act_H2_3g'], tends['act_H2_3g'], '-s', color=colors[1], label=r'$\mathrm{H}_2$, 3 g')
ax.plot(TinsdegC['act_H2_6g'], tends['act_H2_6g'], '-x', color=colors[1], label=r'$\mathrm{H}_2$, 6 g', markersize=10)
ax.plot(TinsdegC['act_CH4'],   tends['act_CH4'],   '-o', color=colors[2], label=r'$\mathrm{CH}_4$, 15 g')
ax.plot(TinsdegC['act_syngas'],tends['act_syngas'],'-o', color=colors[3], label=r'syngas, 15 g')

# ax.legend()
ax.set(
    xlabel = r'$T_\mathrm{inlet}, \degree \mathrm{C}$',
    ylabel = r'$t_\mathrm{total} $, s',
    ylim = [0,None],
    # xlim=[None, 1150]
)
fig.tight_layout()
fig.savefig('plots/labReactor_results_t_vs_T2.pdf')

## Analytical fitted T

In [ ]:
# NOTE: copied from postprocess_conversions.ipynb! Original is there!
def get_analytical(fuel, T, mass):
    if fuel == 'syngas':
        fuel = 'CO'
        xA0 = 0.5  # [-] - mole fraction of fuel at the inlet
    else:
        fuel = fuel
        xA0 = 1.0
    p                   = 101325    # [Pa] - pressure
    chem                = 'act'
    nVolRate_mLn_min    = 900
    T                   = T
    Ro                  = 0.033
    ms                  = mass # [kg] - mass of oxygen carrier
    phase = IlmenitePhase(rhoox=3000, Ro=Ro)
    reaction = ilmenite[chem][fuel]
    oxidations = np.linspace(0,1,121)
    Vdot0 = nVolRate_mLn_min * 1e-6 / 60 * T / ( 25 + 273.15 )   # [m^3/s] - total volumetric gas flow at inlet
    f = np.vectorize(gas_yield_Leion)
    gas_yield = f(phase, reaction, ms, oxidations, T, p, xA0, Vdot0)
    omega = 1 - Ro * (1-oxidations)
    return omega, gas_yield

In [ ]:
# Demo and visual verification that fit is actually a fit

fuel = 'CO'
listTexpdegC = [800,900,950,1000,1050]

for TexpdegC in listTexpdegC:
    x, y = remove_initial_from_validation_data(
        dfs_validation['CO'][f'{TexpdegC}C_X'].dropna().to_numpy(), 
        dfs_validation['CO'][f'{TexpdegC}C_Y'].dropna().to_numpy(), 
        limit_fraction=0.8)

    T = fit_analytical_gamma_omega_T_Berguerand(fuel, x, y, mass=0.003)

    h = plt.plot(x,y)
    xa, ya = get_analytical(fuel, T, 0.003)
    plt.plot(xa, ya, color=h[0]._color)
    plt.gca().invert_xaxis()

In [ ]:
# fuel = 'CO'
# listTexpdegC = [800,900,950,1000,1050]

dTexp = {
    'CO': [800,900,950,1000,1050],
    'H2': [800,900]
}

dTfitdegC = {
    'CO': [],
    'H2': [],
}
for fuel, listTexpdegC in dTexp.items():
    print(fuel)

    arr_TavgdegC = list()
    for TdegC in listTexpdegC:
        print(TdegC)
        if fuel == 'H2' and TdegC == 900:
            key = '900-1050'
        else:
            key = TdegC
        x, y = remove_initial_from_validation_data(
            dfs_validation[fuel][f'{key}C_X'].dropna().to_numpy(), 
            dfs_validation[fuel][f'{key}C_Y'].dropna().to_numpy(), 
            limit_fraction=0.8)

        Tfit = fit_analytical_gamma_omega_T_Berguerand(fuel, x, y, mass=0.003)
        dTfitdegC[fuel].append(Tfit-273.15)
#     print(f'{TdegC:.2f}: {Tfit-273.15:.2f}')

fig, ax = plt.subplots(figsize=[90*mm, 65*mm])
ax.plot(dTexp['CO'], dTfitdegC['CO'], '-o', label=r'CO')
ax.plot(dTexp['H2'], dTfitdegC['H2'], '-o', label=r'$\mathrm{H}_2$')
ax.plot(dTexp['CO'], dTexp['CO'], '--', color='gray')
# ax.grid()
ax.legend()
ax.set(
    xlabel = r'$T_\mathrm{inlet}, ^\circ \mathrm{C}$',
    ylabel = r'$T_\mathrm{fit}, ^\circ \mathrm{C}$',
)
fig.tight_layout()
fig.savefig('labReactor_analytical_fit_of_exp.pdf')

print('Inlet temperature versus temperature of analytical curve, fitted to experimental data corresponding to that inlet temperature')
# no simulation data is used here

In [ ]:
# fuel = 'CO'
# listTexpdegC = [800,900,950,1000,1050]

dTexp = {
    'CO': [800,900,950,1000,1050],
    'H2': [800,900,1050]
}

dTfitdegC = {
    'CO': [],
    'H2': [],
}
for fuel, listTexpdegC in dTexp.items():
    print(fuel)

    arr_TavgdegC = list()
    for TdegC in listTexpdegC:
        print(TdegC)
        if fuel == 'H2' and (TdegC in [900,1050]):
            key = '900-1050'
        else:
            key = TdegC
        x, y = remove_initial_from_validation_data(
            dfs_validation[fuel][f'{key}C_X'].dropna().to_numpy(), 
            dfs_validation[fuel][f'{key}C_Y'].dropna().to_numpy(), 
            limit_fraction=0.8)

        Tfit = fit_analytical_gamma_omega_T_Berguerand(fuel, x, y, mass=0.003)
        dTfitdegC[fuel].append(Tfit-273.15)
#     print(f'{TdegC:.2f}: {Tfit-273.15:.2f}')

fig, ax = plt.subplots(figsize=[90*mm, 65*mm])
ax.plot(dTexp['CO'], dTfitdegC['CO']-np.array(dTexp['CO']), '-o', label=r'CO')
ax.plot(dTexp['H2'], dTfitdegC['H2']-np.array(dTexp['H2']), '-o', label=r'$\mathrm{H}_2$')
# ax.plot(dTexp['CO'], dTexp['CO'], '--', color='gray')
# ax.grid()
ax.legend()
ax.set(
    xlabel = r'$T_\mathrm{inlet}, \mathrm{K}$',
    ylabel = r'$T_\mathrm{fit} - T_\mathrm{inlet}, ^\circ \mathrm{C}$',
)
fig.tight_layout()
fig.savefig('labReactor_analytical_fit_of_exp.pdf')

print('Inlet temperature versus temperature of analytical curve, fitted to experimental data corresponding to that inlet temperature')
# no simulation data is used here